# 11b — Mejoras marginales, juntas (Paso 5)

En los pasos anteriores quedaron **seis** mejoras que, cada una por separado, ganan alrededor de un desvío o menos y no pasaron el criterio:

| Código | Qué es | Notebook |
|---|---|---|
| T3 | Texto de `warranty` dentro del modelo de texto (`PrediccionOOF`) | `09_texto` |
| N | 4 features de fotos | `11_anidadas` |
| L | `hora_inicio`, `gap_actualizacion` | `07_nuevas_features` |
| M | 5 flags de redondez del precio | `07_nuevas_features` |
| V1 | `seller_pct_new` con `TargetEncoder` en vez de *leave-one-out* | `10_vendedor_v2` |
| V2 | Conteo de publicaciones del vendedor en train + test | `10_vendedor_v2` |

Todas van en la misma dirección. Si son señales reales pero chicas, **juntas** deberían sumar algo que se distinga del ruido. Plan:

1. **Base (T2a)** = 0,9291.
2. **Las 6 juntas** y **las 5 sin V2** (V2 no serviría para una publicación que llega sola; ver `10_vendedor_v2`), con comparación pareada contra la Base.
3. **Sacar de a una** (*ablation*): partiendo de las 6 juntas, saco una por vez. Si al sacarla la accuracy **no baja** de forma consistente, esa mejora no estaba aportando y sobra.

Comparación pareada y criterio, igual que en `10_vendedor_v2.ipynb`: **mejora real** si gana en 5 de 5 folds, o en 4 de 5 con la media de la diferencia mayor que su desvío.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

from features import aplanar_anidadas, agregar_logs, CapeoPorMoneda, VendedorEncoder, PrediccionOOF

SEED = 42

train = pd.read_json("../data/train_data.jsonlines", lines=True)
test = pd.read_json("../data/test_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train = aplanar_anidadas(train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

modelo_fijo = HistGradientBoostingClassifier(
    learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED
)

columnas_categoricas_base = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_numericas_base = [
    "log_price", "log_available_quantity", "automatic_relist",
    "sold_quantity", "initial_quantity",
    "n_fotos", "n_atributos", "n_variaciones", "tiene_video", "tiene_garantia",
    "free_shipping", "local_pick_up", "n_metodos_pago", "accepts_mercadopago",
    "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item",
]

resultados_bloques = []
accuracy_por_fold = {}


def evaluar(nombre, pipeline):
    """Mide cualquier pipeline con la CV fija. Guarda la media en resultados_bloques
    y la accuracy de cada fold en accuracy_por_fold (para la comparación pareada)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resultados = cross_validate(pipeline, train, y, cv=cv, scoring=["accuracy", "roc_auc"])

    accuracy_por_fold[nombre] = resultados["test_accuracy"]
    fila = {
        "bloque": nombre,
        "accuracy_media": resultados["test_accuracy"].mean(),
        "accuracy_std": resultados["test_accuracy"].std(),
        "roc_auc_media": resultados["test_roc_auc"].mean(),
    }
    resultados_bloques.append(fila)
    print(nombre, "-> accuracy:", round(fila["accuracy_media"], 4), "+/-", round(fila["accuracy_std"], 4),
          "| roc_auc:", round(fila["roc_auc_media"], 4))


def comparar_pareado(nombre_a, nombre_b):
    """Diferencia de accuracy fold por fold (B - A), sobre los mismos folds."""
    diferencias = accuracy_por_fold[nombre_b] - accuracy_por_fold[nombre_a]
    print(f"{nombre_b}  vs  {nombre_a}")
    print("  diferencia por fold:", np.round(diferencias, 4))
    print("  folds en que gana B:", (diferencias > 0).sum(), "de", len(diferencias))
    print("  media de la diferencia:", round(diferencias.mean(), 4), "| desvío de la diferencia:", round(diferencias.std(), 4))

## Features determinísticas de cada mejora

Todas se calculan una vez, afuera del `Pipeline`, porque no aprenden nada de los datos. Es el **mismo código** que en la notebook de origen de cada una:

- **T3** (`09_texto`): `warranty` nula → texto vacío.
- **N** (`11_anidadas`): resolución de la primera foto, % cuadradas, % con proporción de cámara, cantidad de tamaños distintos.
- **L** (`07_nuevas_features`): hora de `start_time` y días entre `start_time` y `last_updated`.
- **M** (`07_nuevas_features`): 5 flags de redondez sobre `price` **sin capear**.
- **V2** (`10_vendedor_v2`): log(1 + publicaciones del vendedor en train + test).

In [2]:
# T3
train["warranty_texto"] = train["warranty"].fillna("")


# N
def medidas_fotos(lista_fotos):
    """Ancho y alto (en píxeles) de cada foto, a partir de max_size "ANCHOxALTO"."""
    medidas = []
    for foto in lista_fotos:
        try:
            ancho, alto = foto["max_size"].split("x")
            medidas.append((int(ancho), int(alto)))
        except (KeyError, AttributeError, ValueError):
            continue
    return medidas


def es_proporcion_camara(ancho, alto):
    return abs(ancho / alto - 4 / 3) < 0.01 or abs(ancho / alto - 3 / 4) < 0.01


medidas = train["pictures"].map(medidas_fotos)
train["foto_log_px_primera"] = medidas.map(lambda m: np.log(m[0][0] * m[0][1]) if m else np.nan)
train["foto_pct_cuadradas"] = medidas.map(lambda m: np.mean([a == h for a, h in m]) if m else np.nan)
train["foto_pct_camara"] = medidas.map(lambda m: np.mean([es_proporcion_camara(a, h) for a, h in m]) if m else np.nan)
train["foto_n_tamanios"] = medidas.map(lambda m: len(set(m)))

# L
train["last_updated"] = pd.to_datetime(train["last_updated"]).dt.tz_localize(None)
train["hora_inicio"] = train["start_time"].dt.hour
train["gap_actualizacion"] = (train["last_updated"] - train["start_time"]).dt.total_seconds() / 86400

# M
precio = train["price"]
train["tiene_decimales"] = precio % 1 != 0
train["mult_100"] = precio % 100 == 0
train["mult_1000"] = precio % 1000 == 0
train["termina_9"] = precio.astype("int64") % 10 == 9
train["termina_99"] = precio.astype("int64") % 100 == 99

# V2
conteo_total = pd.concat([train["seller_id"], test["seller_id"]]).value_counts()
train["seller_n_total"] = np.log1p(train["seller_id"].map(conteo_total))

columnas_extra_por_codigo = {
    "N": ["foto_log_px_primera", "foto_pct_cuadradas", "foto_pct_camara", "foto_n_tamanios"],
    "L": ["hora_inicio", "gap_actualizacion"],
    "M": ["tiene_decimales", "mult_100", "mult_1000", "termina_9", "termina_99"],
    "V2": ["seller_n_total"],
}

## Un solo constructor de pipelines

`armar_pipeline(codigos)` arma T2a y le suma las mejoras cuyo código esté en `codigos`:

- **T3** cambia el modelo de texto de `PrediccionOOF` por el que también lee `warranty_texto`. Reemplaza `prob_texto_new`, no agrega otra.
- **V1** cambia cómo se codifica `seller_pct_new`: `VendedorEncoder` sin target + `TargetEncoder` sobre `seller_id`.
- **N, L, M, V2** suman sus columnas numéricas.

Con `armar_pipeline(set())` sale exactamente T2a.

In [3]:
def armar_modelo_texto(C, usar_garantia=False):
    vectores = [
        ("palabras", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"), "title"),
        ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                                       strip_accents="unicode"), "title"),
    ]
    if usar_garantia:
        vectores.append(("garantia", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"),
                         "warranty_texto"))
    return Pipeline([
        ("vectores", ColumnTransformer(vectores)),
        ("modelo", LogisticRegression(C=C, max_iter=1000, random_state=SEED)),
    ])


def armar_pipeline(codigos):
    columnas_numericas = list(columnas_numericas_base)
    for codigo in ["N", "L", "M", "V2"]:
        if codigo in codigos:
            columnas_numericas += columnas_extra_por_codigo[codigo]

    if "T3" in codigos:
        texto = ("texto", PrediccionOOF(armar_modelo_texto(C=2, usar_garantia=True), random_state=SEED),
                 ["title", "warranty_texto"])
    else:
        texto = ("texto", PrediccionOOF(armar_modelo_texto(C=2), random_state=SEED), ["title"])

    transformers = [
        ("onehot", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas_base),
        ("num", "passthrough", columnas_numericas),
        ("categoria", TargetEncoder(target_type="binary", random_state=SEED), ["category_id"]),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        texto,
    ]
    if "V1" in codigos:
        transformers.append(("vendedor", VendedorEncoder(incluir_target=False), ["seller_id", "log_price"]))
        transformers.append(("vendedor_te", TargetEncoder(target_type="binary", random_state=SEED), ["seller_id"]))
    else:
        transformers.append(("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]))

    return Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", ColumnTransformer(transformers, sparse_threshold=0)),
        ("modelo", modelo_fijo),
    ])


TODAS = {"T3", "N", "L", "M", "V1", "V2"}

## Base (T2a)

Tiene que dar 0,9291.

In [4]:
evaluar("Base (T2a)", armar_pipeline(set()))

Base (T2a) -> accuracy: 0.9291 +/- 0.0017 | roc_auc: 0.9807


## Las 6 juntas, y las 5 sin V2

In [5]:
evaluar("Todas (6)", armar_pipeline(TODAS))
comparar_pareado("Base (T2a)", "Todas (6)")

Todas (6) -> accuracy: 0.9342 +/- 0.0017 | roc_auc: 0.983
Todas (6)  vs  Base (T2a)
  diferencia por fold: [0.0053 0.0044 0.007  0.0034 0.0056]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0051 | desvío de la diferencia: 0.0012


In [6]:
evaluar("Todas sin V2 (5)", armar_pipeline(TODAS - {"V2"}))
comparar_pareado("Base (T2a)", "Todas sin V2 (5)")

Todas sin V2 (5) -> accuracy: 0.9337 +/- 0.0019 | roc_auc: 0.9827
Todas sin V2 (5)  vs  Base (T2a)
  diferencia por fold: [0.0059 0.0031 0.0061 0.0033 0.0045]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0046 | desvío de la diferencia: 0.0013


## Sacar de a una (*ablation*)

Parto de las 6 juntas y saco una por vez. Ahora la comparación pareada es **"Todas (6)" contra "Todas sin X"**:

- Si al sacar X la accuracy **baja** de forma consistente (las 6 ganan en 5 de 5 folds, o en 4 de 5 con media > desvío), X **aporta** dentro del conjunto.
- Si no baja (o baja solo en algunos folds), X **sobra**: se puede sacar sin perder nada y el modelo queda más simple.

In [7]:
for codigo in ["T3", "N", "L", "M", "V1", "V2"]:
    nombre = f"Todas sin {codigo}"
    if codigo == "V2":
        nombre = "Todas sin V2 (5)"
    else:
        evaluar(nombre, armar_pipeline(TODAS - {codigo}))
    comparar_pareado(nombre, "Todas (6)")
    print()

Todas sin T3 -> accuracy: 0.9332 +/- 0.0007 | roc_auc: 0.9826
Todas (6)  vs  Todas sin T3
  diferencia por fold: [ 0.0004 -0.0009  0.0024  0.0006  0.0026]
  folds en que gana B: 4 de 5
  media de la diferencia: 0.001 | desvío de la diferencia: 0.0013



Todas sin N -> accuracy: 0.9328 +/- 0.0016 | roc_auc: 0.9825
Todas (6)  vs  Todas sin N
  diferencia por fold: [0.0009 0.0011 0.003  0.0003 0.0021]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0015 | desvío de la diferencia: 0.001



Todas sin L -> accuracy: 0.9325 +/- 0.0013 | roc_auc: 0.9824
Todas (6)  vs  Todas sin L
  diferencia por fold: [0.0015 0.0009 0.0029 0.0006 0.0027]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0017 | desvío de la diferencia: 0.0009



Todas sin M -> accuracy: 0.9339 +/- 0.002 | roc_auc: 0.9828
Todas (6)  vs  Todas sin M
  diferencia por fold: [-0.0006  0.0009  0.0016 -0.0015  0.0013]
  folds en que gana B: 3 de 5
  media de la diferencia: 0.0003 | desvío de la diferencia: 0.0012



Todas sin V1 -> accuracy: 0.9341 +/- 0.0015 | roc_auc: 0.983
Todas (6)  vs  Todas sin V1
  diferencia por fold: [ 0.0006 -0.0005  0.0009 -0.0001 -0.0001]
  folds en que gana B: 2 de 5
  media de la diferencia: 0.0001 | desvío de la diferencia: 0.0005

Todas (6)  vs  Todas sin V2 (5)
  diferencia por fold: [-0.0006  0.0012  0.0009  0.0001  0.0011]
  folds en que gana B: 4 de 5
  media de la diferencia: 0.0005 | desvío de la diferencia: 0.0007



## Tabla final y conclusión

In [8]:
tabla_marginales = pd.DataFrame(resultados_bloques).set_index("bloque")
base = accuracy_por_fold["Base (T2a)"]
tabla_marginales["ganancia_vs_base"] = tabla_marginales["accuracy_media"] - base.mean()
tabla_marginales["folds_mejor_que_base"] = [(accuracy_por_fold[nombre] > base).sum() for nombre in tabla_marginales.index]
tabla_marginales.round(4)

,accuracy_media,accuracy_std,roc_auc_media,ganancia_vs_base,folds_mejor_que_base
bloque,,,,,
Base (T2a),0.9291,0.0017,0.9807,0.0000,0
Todas (6),0.9342,0.0017,0.9830,0.0051,5
Todas sin V2 (5),0.9337,0.0019,0.9827,0.0046,5
Todas sin T3,0.9332,0.0007,0.9826,0.0041,5
Todas sin N,0.9328,0.0016,0.9825,0.0037,5
Todas sin L,0.9325,0.0013,0.9824,0.0034,5
Todas sin M,0.9339,0.0020,0.9828,0.0048,5
Todas sin V1,0.9341,0.0015,0.9830,0.0050,5


## Confirmación: el subconjunto que aporta, medido directo

Que al sacar N o L de las 6 la accuracy baje **no garantiza** que N + L solas rindan lo mismo que las 6 (las mejoras pueden interactuar). Por eso mido directo, contra la Base y contra las 6 juntas:

- **N + L**: las dos que pasaron el criterio en el *ablation*.
- **N + L + T3**: T3 quedó en el borde (4 de 5 folds, media +0,0010 < desvío 0,0013).

In [7]:
evaluar("N + L", armar_pipeline({"N", "L"}))
comparar_pareado("Base (T2a)", "N + L")
comparar_pareado("N + L", "Todas (6)")

N + L -> accuracy: 0.9329 +/- 0.0016 | roc_auc: 0.982
N + L  vs  Base (T2a)
  diferencia por fold: [0.0047 0.0031 0.0044 0.003  0.0039]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0038 | desvío de la diferencia: 0.0007
Todas (6)  vs  N + L
  diferencia por fold: [0.0006 0.0013 0.0026 0.0004 0.0017]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0013 | desvío de la diferencia: 0.0008


In [8]:
evaluar("N + L + T3", armar_pipeline({"N", "L", "T3"}))
comparar_pareado("Base (T2a)", "N + L + T3")
comparar_pareado("N + L", "N + L + T3")
comparar_pareado("N + L + T3", "Todas (6)")

N + L + T3 -> accuracy: 0.9339 +/- 0.0018 | roc_auc: 0.9827
N + L + T3  vs  Base (T2a)
  diferencia por fold: [0.0048 0.0039 0.0075 0.0034 0.0045]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0048 | desvío de la diferencia: 0.0014
N + L + T3  vs  N + L
  diferencia por fold: [0.0001 0.0009 0.0031 0.0004 0.0006]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.001 | desvío de la diferencia: 0.0011
Todas (6)  vs  N + L + T3
  diferencia por fold: [ 0.0005  0.0004 -0.0005  0.0001  0.0011]
  folds en que gana B: 4 de 5
  media de la diferencia: 0.0003 | desvío de la diferencia: 0.0005


In [9]:
tabla_confirmacion = pd.DataFrame(resultados_bloques).set_index("bloque").loc[["Base (T2a)", "Todas (6)", "N + L", "N + L + T3"]]
tabla_confirmacion["ganancia_vs_base"] = tabla_confirmacion["accuracy_media"] - base.mean()
tabla_confirmacion["folds_mejor_que_base"] = [(accuracy_por_fold[nombre] > base).sum() for nombre in tabla_confirmacion.index]
tabla_confirmacion.round(4)

,accuracy_media,accuracy_std,roc_auc_media,ganancia_vs_base,folds_mejor_que_base
bloque,,,,,
Base (T2a),0.9291,0.0017,0.9807,0.0000,0
Todas (6),0.9342,0.0017,0.9830,0.0051,5
N + L,0.9329,0.0016,0.9820,0.0038,5
N + L + T3,0.9339,0.0018,0.9827,0.0048,5


**Conclusión.**

| Configuración | accuracy | roc_auc | vs Base (folds) |
|---|---|---|---|
| Base (T2a) | 0,9291 ± 0,0017 | 0,9807 | — |
| Todas (6) | 0,9342 ± 0,0017 | 0,9830 | 5/5, +0,0051 |
| N + L | 0,9329 ± 0,0016 | 0,9820 | 5/5, +0,0038 |
| **N + L + T3** | **0,9339 ± 0,0018** | **0,9827** | **5/5, +0,0048** |

1. **Juntas, las marginales sí suman.** Las 6 dan +0,0051, ganan en los 5 folds y la media de la diferencia es 4 veces su desvío. Señales chicas pero reales, que por separado no se distinguían del ruido.
2. ***Ablation* (sacar de a una de las 6):**
   - **Aportan**: **N** (fotos, 5/5, +0,0015) y **L** (fechas, 5/5, +0,0017).
   - **En el borde**: **T3** (garantía, 4/5, +0,0010 < desvío 0,0013).
   - **Sobran**: **M** (redondez del precio, 3/5), **V1** (`TargetEncoder` del vendedor, 2/5) y **V2** (conteo train + test, 4/5, +0,0005 < 0,0007).
3. **Confirmación directa:**
   - N + L solas ganan a la Base en 5/5 (+0,0038).
   - Sumarles **T3 mejora en 5 de 5 folds** (+0,0010), así que T3 sí aporta en este conjunto.
   - Las 6 contra N + L + T3: 4/5 y +0,0003 < desvío 0,0005, o sea que **M, V1 y V2 no agregan nada**.

**Configuración ganadora del Paso 5: T2a + N + L + T3 = 0,9339 ± 0,0018 / 0,9827.** Es la de 6 sin las tres que sobran: prácticamente igual de buena (−0,0003, ruido) y más simple. Además deja afuera V2, que no serviría en producción.

Lectura (interpretación, no medida): M probablemente se solapa con el texto y con el vendedor (un precio "de tienda" viene con título y vendedor de tienda). V1 y V2 tocan el vendedor, que ya estaba bien cubierto. N, L y T3 agregan información de **contenido y comportamiento** de la publicación (cómo son las fotos, cuándo se publicó y actualizó, qué dice la garantía), que es justo lo que falta para los vendedores sin historial (`08_errores`).
